# Week 10 Pre-Lab: SAR & Sensor Fusion — ARIA v7.0 Setup

**Course:** NTU Remote Sensing & Spatial Information Analysis（遙測與空間資訊之分析與應用）  
**Instructor:** Prof. Su Wen-Ray  
**Week:** 10  
**Theme:** All-Weather Monitoring & Sensor Fusion  
**Estimated Time:** 約 20–40 分鐘

---

## 本 notebook 會完成什麼？

這份 notebook 將 Week 10 pre-lab 轉成可執行版本，包含：

1. 確認 Week 8/9 的 Python 環境與核心套件是否可用。
2. 安裝或檢查 SAR 相關套件：`rasterio`, `rioxarray`。
3. 理解 SAR 的基本物理概念：backscatter、specular reflection、volume scattering、double-bounce。
4. 完成 SAR vs. Optical 的比較與 dB conversion 自我測試。
5. 建立 Sentinel-1 RTC STAC 串流的範例工作流。
6. 建立 SAR threshold water detection 與 optical-SAR sensor fusion 的作業模板。

> 注意：STAC 串流區塊需要網路，且需能連線至 Microsoft Planetary Computer。若只做課前準備，可以先執行環境檢查與 dB conversion。

## 0. 使用方式

如果你只是要完成 pre-lab，建議依序執行：

1. **Step 1–2：環境與套件檢查**
2. **Step 3–4：SAR 觀念整理**
3. **Step 7：dB Conversion Self-Test**
4. **Checklist Before Class**

如果你要進一步接作業或課堂 lab，再執行後面的：

- Sentinel-1 RTC STAC 搜尋與串流
- SAR water mask
- Optical + SAR sensor fusion

---

## 重要提醒

本週與 Week 8/9 的主要差異：

| 項目 | Week 8/9 Optical | Week 10 SAR |
|---|---|---|
| Collection | `sentinel-2-l2a` | `sentinel-1-rtc` |
| Main assets | `B02`, `B03`, `B04`, `B08`, `B11` | `vv`, `vh` |
| 是否需要除以 10000 | 是 | 否 |
| 是否需要雲量過濾 | 是 | 否 |
| 常見處理 | NDVI / NDWI / BSI | Linear → dB, speckle filtering |
| 水體偵測 | NDWI threshold | VV dB threshold |


# Step 1. Verify Week 8/9 Environment

請先啟動你之前 Week 8/9 使用的環境。例如：

```bash
conda activate remo_w8
```

或如果你用 `venv`：

```bash
source ~/remo_env/bin/activate
```

Windows PowerShell 可能類似：

```powershell
.\remo_env\Scripts\activate
```


In [ ]:
# Step 1a. Optional installation cell
# 如果你已經安裝好套件，保持 INSTALL_PACKAGES = False 即可。
# 若有 import error，再改成 True 執行。

INSTALL_PACKAGES = False

if INSTALL_PACKAGES:
    import sys
    !{sys.executable} -m pip install pystac-client planetary-computer stackstac scikit-learn rasterio rioxarray scipy matplotlib pandas numpy xarray geopandas shapely
else:
    print("INSTALL_PACKAGES = False；略過安裝。若後面 import 失敗，再回來改成 True。")

In [ ]:
# Step 1b. Confirm key packages are available

import importlib
from importlib.metadata import version, PackageNotFoundError

packages_to_check = [
    "numpy",
    "pandas",
    "matplotlib",
    "rasterio",
    "rioxarray",
    "pystac_client",
    "planetary_computer",
    "stackstac",
    "sklearn",
    "scipy",
    "xarray",
]

module_import_names = {
    "pystac_client": "pystac_client",
    "planetary_computer": "planetary_computer",
    "sklearn": "sklearn",
}

print("Package check:")
print("-" * 60)

missing = []
for pkg in packages_to_check:
    import_name = module_import_names.get(pkg, pkg)
    try:
        mod = importlib.import_module(import_name)
        try:
            v = version(pkg.replace("_", "-"))
        except PackageNotFoundError:
            v = getattr(mod, "__version__", "installed")
        print(f"✓ {pkg}: {v}")
    except Exception as e:
        print(f"✗ {pkg}: {type(e).__name__}: {e}")
        missing.append(pkg)

if missing:
    print("\n需要安裝或修復的套件：", missing)
    print("可以回到上一格，把 INSTALL_PACKAGES 改成 True 後再執行。")
else:
    print("\n✓ All core dependencies loaded successfully.")

# Step 2. Install SAR-Specific Packages

本週新增的 SAR 相關套件主要是：

```bash
pip install rasterio rioxarray
```

用途：

| Package | 用途 |
|---|---|
| `rasterio` | 讀取與寫出 GeoTIFF、取得 CRS、transform、metadata |
| `rioxarray` | 讓 `xarray` 支援 raster CRS / clipping / reprojection |

下面這格會再次確認 `rasterio` 與 `rioxarray` 是否能正常載入。

In [ ]:
import rasterio
import rioxarray

print("✓ rasterio:", rasterio.__version__)
print("✓ rioxarray ready for SAR GeoTIFF loading")

# Step 3. SAR Physics — Conceptual Review

## 3.1 What Is SAR?

**SAR（Synthetic Aperture Radar，合成孔徑雷達）** 是一種 **active remote sensing system（主動式遙測系統）**。

它的基本邏輯是：

1. 衛星主動發射 microwave pulses。
2. 地表反射或散射部分能量。
3. 衛星接收返回的訊號，稱為 **backscatter（後向散射）**。

Sentinel-1 使用 C-band microwave，波長約 **5.6 cm**。由於微波能穿透雲層，SAR 可以在：

- 雲多的颱風期間
- 夜晚
- 雨後或災害期間

持續取得地表資訊。

---

## 3.2 Key Backscatter Mechanisms

| Surface Type | Backscatter Mechanism | Signal Strength | Typical dB |
|---|---|---:|---:|
| Calm water | Specular reflection：鏡面反射，能量多被反射到旁邊 | Very low | `< -20 dB` |
| Rough water / wetland | Partial diffuse scattering | Low–Medium | `-15 to -10 dB` |
| Bare soil / urban | Surface scattering | Medium–High | `-10 to -5 dB` |
| Forest / vegetation | Volume scattering：植被內多次散射 | High | `-8 to -3 dB` |
| Buildings | Double-bounce / corner reflector effect | Very high | `> 0 dB` |

---

## 3.3 Why Is SAR Useful for Flood / Water Detection?

洪水或開闊水面通常在 SAR 影像中會很暗，原因是：

- 平靜水面接近鏡面反射。
- 雷達能量被反射到其他方向。
- 回到衛星的 backscatter 很低。

因此常見的簡單規則是：

```text
VV < -18 dB → Water
```

不過，`-18 dB` 只是全球或文獻上常用的初始值。實際案例會受到：

- 水面是否混濁
- 泥沙濃度
- 植被淹水
- 雷達入射角
- 地形陰影
- speckle noise

影響，所以課堂案例可能需要調整 threshold，並搭配 morphological post-processing。

# Step 4. SAR vs. Optical — Comparison Table

請先自己遮住答案，試著完成表格中的空格。

| Feature | Optical（Sentinel-2） | SAR（Sentinel-1） |
|---|---|---|
| Energy source | Sun / passive | Satellite transmitter / active |
| Wavelength | Visible + NIR + SWIR, about 0.4–2.2 μm | Microwave C-band, about 5.6 cm |
| Cloud penetration | ❌ Cannot see through clouds | ✅ Yes |
| Night operation | ❌ Needs sunlight | ✅ Yes |
| Water detection method | NDWI = (Green − NIR) / (Green + NIR) | Backscatter threshold: VV < about -18 dB |
| Vegetation detection | NDVI = (NIR − Red) / (NIR + Red) | Volume scattering, usually high backscatter |
| Spatial resolution | 10 m for B2–B4 and B8 | 10 m for IW GRD / RTC |
| Revisit time | about 5 days with 2-satellite constellation | about 6 days with 2-satellite constellation |

---

## 小結

Optical 影像比較容易解讀地表顏色、植被、水體與裸露地；SAR 則可以在雲雨或夜晚取得資訊。  
因此災害監測中常把兩者結合成 **sensor fusion**，提高判讀穩定性。

# Step 5. Understand the Data You Will Use

## 5.1 課堂 Demo / Lab：STAC API 即時串流

本週會延續 W8–W9 的 STAC workflow，但從 Sentinel-2 換成 Sentinel-1 RTC：

| 步驟 | 說明 | 誰做？ |
|---|---|---|
| STAC 搜尋 `sentinel-1-rtc` | `pystac_client` → `planetary_computer` | 你 |
| 串流讀取 VV / VH band | `stackstac.stack()` → `xarray` | 你 |
| Linear → dB 轉換 | `10 * np.log10(value)` | 你 |
| Speckle filtering | `scipy.ndimage.median_filter(size=5)` | 你 |

**Sentinel-1 RTC** = Radiometrically Terrain Corrected。  
Planetary Computer 已預處理輻射校正與地形校正，所以這份 notebook 主要處理：

1. 讀取資料
2. 轉換成 dB
3. median filter 降低 speckle
4. threshold 偵測水體

---

## 5.2 作業：預處理 GeoTIFF

作業可能會提供預處理好的：

```text
S1_Hualien_dB.tif
```

若你有這份檔案，可以放在 notebook 同資料夾或 `data/` 資料夾，後面的 local GeoTIFF 區塊可以直接讀取。

In [ ]:
# Step 5a. Global configuration

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# STAC settings
# -----------------------------
STAC_ENDPOINT = "https://planetarycomputer.microsoft.com/api/stac/v1"
S1_COLLECTION = "sentinel-1-rtc"
S2_COLLECTION = "sentinel-2-l2a"

# Matai'an / Hualien example AOI, lon_min, lat_min, lon_max, lat_max
# 可依照你的課堂或作業區域調整
MATAIAN_BBOX = [121.28, 23.56, 121.52, 23.76]

# Example dates. 請依老師指定案例調整。
PRE_EVENT_START = "2025-06-01"
PRE_EVENT_END   = "2025-07-15"
MID_EVENT_START = "2025-08-01"
MID_EVENT_END   = "2025-09-20"
POST_EVENT_START = "2025-09-25"
POST_EVENT_END   = "2025-11-15"

# SAR threshold initial value
SAR_WATER_THRESHOLD_DB = -18

# Output folders
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"

for d in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration ready.")
print("AOI bbox:", MATAIAN_BBOX)
print("SAR water threshold:", SAR_WATER_THRESHOLD_DB, "dB")

# Step 6. 課前準備檢查

## 課堂 Lab

課堂中的 Sensor Fusion Lab 預計直接從 STAC 載入 Sentinel-1 / Sentinel-2 資料，因此不一定需要事先準備 W9 的輸出檔案。

## 作業

作業可能需要你沿用 Week 9 的 optical results：

| Item | File / Variable | Status |
|---|---|---|
| Optical water mask | Binary mask from NDWI thresholding | ☐ Ready |
| Cloud mask | SCL-based cloud mask | ☐ Ready |

如果你沒有這些結果，建議先重新跑一次 W9 notebook。

# Step 7. Self-Test — dB Conversion

**Scenario:** 一個 Sentinel-1 pixel 的 backscatter 為：

```text
σ⁰ = 0.001   # linear scale
```

請計算：

1. Convert to dB: `σ⁰_dB = 10 × log10(0.001)`
2. 這個 pixel 比較可能是 water 還是 land？
3. 如果 threshold 是 `-18 dB`，它會被分類為 water 嗎？

In [ ]:
# Step 7a. dB conversion self-test

sigma0_linear = 0.001
sigma0_db = 10 * np.log10(sigma0_linear)
threshold_db = -18
is_water = sigma0_db < threshold_db

print(f"Linear σ⁰ = {sigma0_linear}")
print(f"dB σ⁰ = 10 × log10({sigma0_linear}) = {sigma0_db:.1f} dB")
print("Interpretation:", "water-like low backscatter" if is_water else "land-like higher backscatter")
print(f"Threshold = {threshold_db} dB")
print("Classified as water?", "Yes ✅" if is_water else "No ❌")

## Step 7 Answer

- `10 × log10(0.001) = 10 × (-3) = -30 dB`
- `-30 dB` 是 very low backscatter，所以比較可能是 **water**。
- 若 threshold 是 `-18 dB`，因為 `-30 < -18`，所以會被分類為 **water**。

# Step 8. Sentinel-1 RTC STAC Demo

下面是課堂 lab 可能會用到的 STAC 串流模板。

因為執行這些 cell 需要網路，也可能花較長時間，所以預設：

```python
RUN_STAC_DEMO = False
```

如果你要實際串流資料，把它改成：

```python
RUN_STAC_DEMO = True
```

---

## 工作流

1. 連線到 Planetary Computer STAC。
2. 搜尋 Sentinel-1 RTC items。
3. 使用 `planetary_computer.sign()` 簽署 assets。
4. 使用 `stackstac.stack()` 讀取 `vv` band。
5. 將 linear backscatter 轉成 dB。
6. 套用 median filter 降低 speckle。
7. 用 threshold 產生 water mask。

In [ ]:
# Step 8a. STAC search helper functions

RUN_STAC_DEMO = False  # 要實際跑 STAC demo 時改成 True

def search_stac_items(collection, bbox, start_date, end_date, query=None, max_items=10):
    """Search STAC items from Microsoft Planetary Computer."""
    from pystac_client import Client
    import planetary_computer as pc

    catalog = Client.open(STAC_ENDPOINT)
    search = catalog.search(
        collections=[collection],
        bbox=bbox,
        datetime=f"{start_date}/{end_date}",
        query=query,
        max_items=max_items,
    )
    items = list(search.item_collection())
    signed_items = [pc.sign(item) for item in items]
    return signed_items


def summarize_items(items):
    """Return a small table summarizing STAC items."""
    rows = []
    for item in items:
        props = item.properties
        rows.append({
            "id": item.id,
            "datetime": props.get("datetime"),
            "platform": props.get("platform"),
            "orbit_state": props.get("sat:orbit_state"),
            "relative_orbit": props.get("sat:relative_orbit"),
        })
    return pd.DataFrame(rows)

print("STAC helper functions ready.")

In [ ]:
# Step 8b. Search Sentinel-1 RTC items

if RUN_STAC_DEMO:
    s1_items = search_stac_items(
        collection=S1_COLLECTION,
        bbox=MATAIAN_BBOX,
        start_date=POST_EVENT_START,
        end_date=POST_EVENT_END,
        max_items=10,
    )
    display(summarize_items(s1_items))
    print(f"Found {len(s1_items)} Sentinel-1 RTC items.")
else:
    print("RUN_STAC_DEMO = False；略過 Sentinel-1 RTC 搜尋。")

In [ ]:
# Step 8c. Stack Sentinel-1 RTC VV band with stackstac

import warnings
warnings.filterwarnings("ignore")


def stack_s1_vv(items, bbox, epsg=32651, resolution=10):
    """Stack Sentinel-1 RTC VV band into an xarray DataArray.
    
    Returns
    -------
    xarray.DataArray
        Dimensions usually include time, band, y, x.
    """
    import stackstac

    if len(items) == 0:
        raise ValueError("No STAC items provided.")

    arr = stackstac.stack(
        items,
        assets=["vv"],
        bounds_latlon=bbox,
        epsg=epsg,
        resolution=resolution,
        chunksize=2048,
    )
    return arr


def linear_to_db(x, eps=1e-10):
    """Convert linear SAR backscatter to dB."""
    return 10 * np.log10(np.maximum(x, eps))


def median_speckle_filter_2d(arr2d, size=5):
    """Apply median filter to a 2D array to reduce SAR speckle noise."""
    from scipy.ndimage import median_filter
    return median_filter(arr2d, size=size)


def classify_water_sar_db(vv_db, threshold_db=-18):
    """Classify water from VV dB using a threshold."""
    return vv_db < threshold_db

print("SAR processing functions ready.")

In [ ]:
# Step 8d. Stream, convert to dB, filter, classify water

if RUN_STAC_DEMO:
    if "s1_items" not in globals() or len(s1_items) == 0:
        raise RuntimeError("請先執行 STAC search cell，並確認 s1_items 有資料。")

    s1_stack = stack_s1_vv(s1_items[:1], MATAIAN_BBOX, epsg=32651, resolution=10)
    print(s1_stack)

    # 取第一景、第一個 band = VV
    vv_linear = s1_stack.isel(time=0, band=0).compute()
    vv_db = linear_to_db(vv_linear)

    # median filter expects numpy array
    vv_db_filtered_np = median_speckle_filter_2d(vv_db.values, size=5)
    sar_water_mask_np = classify_water_sar_db(vv_db_filtered_np, threshold_db=SAR_WATER_THRESHOLD_DB)

    print("VV dB range:", np.nanmin(vv_db_filtered_np), np.nanmax(vv_db_filtered_np))
    print("SAR water pixels:", int(np.nansum(sar_water_mask_np)))
else:
    print("RUN_STAC_DEMO = False；略過 STAC 串流處理。")

In [ ]:
# Step 8e. Visualize STAC SAR result

if RUN_STAC_DEMO and "vv_db_filtered_np" in globals():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    im0 = axes[0].imshow(vv_db_filtered_np, cmap="gray", vmin=-25, vmax=0)
    axes[0].set_title("Sentinel-1 RTC VV dB\nMedian filtered")
    axes[0].axis("off")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="dB")

    im1 = axes[1].imshow(sar_water_mask_np, cmap="Blues")
    axes[1].set_title(f"SAR Water Mask\nVV < {SAR_WATER_THRESHOLD_DB} dB")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("沒有可視化資料。若要顯示 STAC SAR 結果，請把 RUN_STAC_DEMO 改成 True 並執行前面 cell。")

# Step 9. Local GeoTIFF Workflow for Homework

作業可能提供已經預處理好的 SAR GeoTIFF，例如：

```text
S1_Hualien_dB.tif
```

請把檔案放在：

```text
data/S1_Hualien_dB.tif
```

或修改下一格的 `S1_LOCAL_PATH`。

In [ ]:
# Step 9a. Load local SAR GeoTIFF

import os
import rasterio
from rasterio.plot import show

S1_LOCAL_PATH = DATA_DIR / "S1_Hualien_dB.tif"

if S1_LOCAL_PATH.exists():
    with rasterio.open(S1_LOCAL_PATH) as src:
        s1_db = src.read(1).astype("float32")
        s1_profile = src.profile.copy()
        s1_crs = src.crs
        s1_transform = src.transform
        nodata = src.nodata

    if nodata is not None:
        s1_db = np.where(s1_db == nodata, np.nan, s1_db)

    print("Loaded:", S1_LOCAL_PATH)
    print("Shape:", s1_db.shape)
    print("CRS:", s1_crs)
    print("dB range:", np.nanmin(s1_db), np.nanmax(s1_db))
else:
    print(f"找不到 {S1_LOCAL_PATH}")
    print("請把 S1_Hualien_dB.tif 放進 data/ 資料夾，或修改 S1_LOCAL_PATH。")
    print("目前 data/ 內的檔案：")
    for p in sorted(DATA_DIR.glob("*")):
        print(" -", p.name)

In [ ]:
# Step 9b. Visualize local SAR GeoTIFF

if "s1_db" in globals():
    plt.figure(figsize=(8, 6))
    im = plt.imshow(s1_db, cmap="gray", vmin=-25, vmax=0)
    plt.title("Local Sentinel-1 VV dB")
    plt.axis("off")
    plt.colorbar(im, label="dB")
    plt.tight_layout()
    plt.show()
else:
    print("尚未載入 s1_db。")

In [ ]:
# Step 9c. SAR water detection from local GeoTIFF

LOCAL_SAR_THRESHOLD_DB = -18  # 可依照 histogram 或老師指定調整

if "s1_db" in globals():
    # 可選：median filter 降低 speckle
    s1_db_filtered = median_speckle_filter_2d(s1_db, size=5)
    sar_water_local = classify_water_sar_db(s1_db_filtered, threshold_db=LOCAL_SAR_THRESHOLD_DB)

    # 排除 NaN
    sar_water_local = np.where(np.isnan(s1_db_filtered), False, sar_water_local)

    print("Threshold:", LOCAL_SAR_THRESHOLD_DB, "dB")
    print("Water pixels:", int(sar_water_local.sum()))
    print("Total valid pixels:", int(np.isfinite(s1_db_filtered).sum()))
    print("Water percentage:", f"{sar_water_local.sum() / np.isfinite(s1_db_filtered).sum() * 100:.2f}%")
else:
    print("尚未載入 s1_db。")

In [ ]:
# Step 9d. Plot local SAR water mask

if "sar_water_local" in globals():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    im0 = axes[0].imshow(s1_db_filtered, cmap="gray", vmin=-25, vmax=0)
    axes[0].set_title("SAR VV dB, median filtered")
    axes[0].axis("off")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="dB")

    axes[1].imshow(sar_water_local, cmap="Blues")
    axes[1].set_title(f"SAR Water Mask\nVV < {LOCAL_SAR_THRESHOLD_DB} dB")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("尚未建立 sar_water_local。")

# Step 10. Sensor Fusion Template

Sensor fusion 的核心概念：

| Optical result | SAR result | Interpretation |
|---|---|---|
| Water | Water | High confidence water |
| Water | Not water | Optical-only water; possible cloud/shadow/threshold issue or SAR rough water issue |
| Not water | Water | SAR-only water; possible cloud-covered flood, shadow, or optical miss |
| Not water | Not water | No water detection |

在 disaster monitoring 中，可以把結果分成：

1. **High confidence area**：Optical 與 SAR 都偵測為 water。
2. **Low confidence / disagreement area**：只有一個 sensor 偵測為 water。
3. **No detection area**：兩者都沒有偵測為 water。

---

## 作業輸入建議

如果你有 Week 9 的 optical water mask，可以存成：

```text
data/optical_water_mask.npy
```

如果你有 cloud mask，可以存成：

```text
data/cloud_mask.npy
```

下面會示範如何讀取並進行 fusion。

In [ ]:
# Step 10a. Load optical water mask and cloud mask if available

OPTICAL_WATER_PATH = DATA_DIR / "optical_water_mask.npy"
CLOUD_MASK_PATH = DATA_DIR / "cloud_mask.npy"

optical_water = None
cloud_mask = None

if OPTICAL_WATER_PATH.exists():
    optical_water = np.load(OPTICAL_WATER_PATH).astype(bool)
    print("Loaded optical water mask:", OPTICAL_WATER_PATH, optical_water.shape)
else:
    print("找不到 optical_water_mask.npy。若要做 fusion，請先輸出或放入 Week 9 optical water mask。")

if CLOUD_MASK_PATH.exists():
    cloud_mask = np.load(CLOUD_MASK_PATH).astype(bool)
    print("Loaded cloud mask:", CLOUD_MASK_PATH, cloud_mask.shape)
else:
    print("找不到 cloud_mask.npy。若沒有 cloud mask，可先略過。")

In [ ]:
# Step 10b. Sensor fusion function

def fuse_optical_sar(optical_water, sar_water, cloud_mask=None):
    """Fuse optical water mask and SAR water mask.

    Classes:
    0 = No detection
    1 = SAR only water
    2 = Optical only water
    3 = Both sensors water, high confidence
    4 = Cloud-masked optical area and SAR water, useful all-weather evidence
    """
    if optical_water.shape != sar_water.shape:
        raise ValueError(
            f"Shape mismatch: optical {optical_water.shape}, SAR {sar_water.shape}. "
            "請先確保兩者已經對齊到相同 CRS、resolution、extent。"
        )

    optical_water = optical_water.astype(bool)
    sar_water = sar_water.astype(bool)

    fusion = np.zeros(optical_water.shape, dtype=np.uint8)
    fusion[(sar_water) & (~optical_water)] = 1
    fusion[(~sar_water) & (optical_water)] = 2
    fusion[(sar_water) & (optical_water)] = 3

    if cloud_mask is not None:
        if cloud_mask.shape != optical_water.shape:
            raise ValueError("cloud_mask shape must match optical_water and sar_water.")
        # 若光學被雲遮住，但 SAR 偵測到水，標記為 4
        fusion[(cloud_mask.astype(bool)) & (sar_water)] = 4

    return fusion


def summarize_fusion(fusion, pixel_area_m2=100):
    """Summarize fusion class counts and area.
    Default pixel area = 10 m × 10 m = 100 m².
    """
    labels = {
        0: "No detection",
        1: "SAR only water",
        2: "Optical only water",
        3: "Both sensors water / high confidence",
        4: "Cloud-masked optical + SAR water",
    }
    rows = []
    for cls, label in labels.items():
        count = int(np.sum(fusion == cls))
        area_km2 = count * pixel_area_m2 / 1_000_000
        rows.append({
            "class": cls,
            "label": label,
            "pixel_count": count,
            "area_km2": area_km2,
        })
    return pd.DataFrame(rows)

print("Fusion functions ready.")

In [ ]:
# Step 10c. Run fusion if masks are available

if optical_water is not None and "sar_water_local" in globals():
    try:
        fusion_map = fuse_optical_sar(optical_water, sar_water_local, cloud_mask=cloud_mask)
        fusion_summary = summarize_fusion(fusion_map, pixel_area_m2=100)
        display(fusion_summary)
    except ValueError as e:
        print("Fusion 無法執行：")
        print(e)
        print("常見原因：optical mask 與 SAR mask 的 shape 不同，需先 reproject/resample/clip 到同一格網。")
else:
    print("尚未同時取得 optical_water 與 sar_water_local，因此略過 fusion。")

In [ ]:
# Step 10d. Plot fusion map

if "fusion_map" in globals():
    plt.figure(figsize=(8, 6))
    im = plt.imshow(fusion_map, vmin=0, vmax=4)
    plt.title("Optical + SAR Sensor Fusion Map")
    plt.axis("off")
    cbar = plt.colorbar(im, ticks=[0, 1, 2, 3, 4])
    cbar.ax.set_yticklabels([
        "No detection",
        "SAR only",
        "Optical only",
        "Both / high confidence",
        "Cloud + SAR water",
    ])
    plt.tight_layout()
    plt.show()
else:
    print("尚未建立 fusion_map。")

In [ ]:
# Step 10e. Save outputs

if "sar_water_local" in globals() and "s1_profile" in globals():
    out_sar_mask = OUTPUT_DIR / "sar_water_mask.tif"

    profile = s1_profile.copy()
    profile.update(
        dtype=rasterio.uint8,
        count=1,
        nodata=255,
        compress="lzw",
    )

    sar_mask_to_save = sar_water_local.astype("uint8")

    with rasterio.open(out_sar_mask, "w", **profile) as dst:
        dst.write(sar_mask_to_save, 1)

    print("Saved:", out_sar_mask)

if "fusion_map" in globals() and "s1_profile" in globals():
    out_fusion = OUTPUT_DIR / "sensor_fusion_map.tif"

    profile = s1_profile.copy()
    profile.update(
        dtype=rasterio.uint8,
        count=1,
        nodata=255,
        compress="lzw",
    )

    with rasterio.open(out_fusion, "w", **profile) as dst:
        dst.write(fusion_map.astype("uint8"), 1)

    print("Saved:", out_fusion)

if "fusion_summary" in globals():
    out_summary = OUTPUT_DIR / "sensor_fusion_summary.csv"
    fusion_summary.to_csv(out_summary, index=False, encoding="utf-8-sig")
    print("Saved:", out_summary)

# Step 11. Reflection Questions

請用 2–3 句回答每一題。可以用中文，也可以中英混合。

---

## Q1. Why can't we just use SAR instead of optical?

**Answer:**  
SAR 的優點是可以穿透雲層、夜間也能觀測，因此很適合颱風、豪雨或災害期間的 all-weather monitoring。可是 SAR 主要反映的是地表粗糙度、含水狀態與散射機制，無法像 optical 影像一樣直接提供顏色、植被光譜、NDVI、NDWI 或裸露地變化等資訊。因此 SAR 不能完全取代 optical，而是應該與 optical 互補使用。

---

## Q2. Speckle noise: Why do SAR images look grainy compared to optical?

**Answer:**  
SAR 使用 coherent illumination，回波訊號會因為地表多個散射體的 constructive interference 與 destructive interference 產生明暗斑點。這種顆粒狀雜訊稱為 speckle noise，是 SAR 影像常見特徵，因此常需要 median filter 或 multi-looking 等方式降低雜訊。

---

## Q3. Sensor fusion logic: If optical says water AND SAR says water → high confidence. What if they disagree?

**Answer:**  
如果 optical 與 SAR 不一致，可能代表其中一種 sensor 受到限制，例如 optical 受到 cloud、shadow 或 threshold 影響；SAR 可能受到 rough water、vegetation flooding、speckle noise、terrain shadow 或 double-bounce 影響。實務上可以把 disagreement area 視為 low confidence area，需要搭配時間序列、現地資料或人工判讀再確認。

# Checklist Before Class

請確認你已完成：

- [ ] Verified `rasterio` and `rioxarray` are installed
- [ ] Reviewed SAR backscatter mechanisms: specular, volume, double-bounce
- [ ] Completed SAR vs. Optical comparison table
- [ ] Completed dB conversion self-test
- [ ] Understood why `VV < -18 dB` is only an initial threshold
- [ ] Understood Sentinel-1 RTC workflow from Planetary Computer STAC
- [ ] W9 optical outputs accessible if needed for homework
- [ ] Reflected on sensor fusion disagreement logic

---

## Final Reminder

如果課堂或作業中遇到以下問題：

1. `stackstac` 讀取很慢：先縮小 bbox 或只讀一景。
2. STAC 找不到資料：檢查日期、bbox、collection 名稱。
3. SAR mask 太碎：調整 threshold 或增加 median filter size。
4. Optical/SAR fusion shape mismatch：需要先 reproject/resample/clip 到相同 CRS、解析度與範圍。
5. GeoTIFF 讀不到：確認路徑是否正確，以及檔案是否放在 `data/`。

**You're ready for Week 10.**